In [ ]:
import numpy as np
import copy
import matplotlib.path as mpltPath
from PIL import Image

# from avaframe
def polygon2Raster(demHeader, Line, radius, th=""):
    """convert line to raster

    Parameters
    ----------
    demHeader: dict
        dem header dictionary
    Line : dict
        line dictionary
    radius : float
        include all cells which center is in the polygon or close enough
    th: float
        thickness value ot the line feature

    Returns
    -------
    Mask : 2D numpy array
        updated raster
    """
    # adim and center dem and polygon
    ncols = demHeader["ncols"]
    nrows = demHeader["nrows"]
    xllc = demHeader["xllcenter"]
    yllc = demHeader["yllcenter"]
    csz = demHeader["cellsize"]
    xCoord0 = (Line["x"] - xllc) / csz
    yCoord0 = (Line["y"] - yllc) / csz
    if (xCoord0[0] == xCoord0[-1]) and (yCoord0[0] == yCoord0[-1]):
        xCoord = np.delete(xCoord0, -1)
        yCoord = np.delete(yCoord0, -1)
    else:
        xCoord = copy.deepcopy(xCoord0)
        yCoord = copy.deepcopy(yCoord0)

    # get the raster corresponding to the polygon
    polygon = np.stack((xCoord, yCoord), axis=-1)
    path = path = mpltPath.Path(polygon)
    # add a tolerance to include cells for which the center is on the lines
    # for this we need to know if the path is clockwise or counterclockwise
    # to decide if the radius should be positive or negative in contains_points
    is_ccw = isCounterClockWise(path)
    r = radius * is_ccw - radius * (1 - is_ccw)
    x = np.linspace(0, ncols - 1, ncols)
    y = np.linspace(0, nrows - 1, nrows)
    X, Y = np.meshgrid(x, y)
    X = X.flatten()
    Y = Y.flatten()
    points = np.stack((X, Y), axis=-1)
    mask = path.contains_points(points, radius=r)
    Mask = mask.reshape((nrows, ncols)).astype(int)
    # thickness field is provided, then return array with ones
    if th != "":
        Mask = np.where(Mask > 0, th, 0.0)
    else:
        Mask = np.where(Mask > 0, 1.0, 0.0)

    return Mask

def isCounterClockWise(path):
    """Determines if a polygon path is mostly clockwise or counter clockwise

    https://stackoverflow.com/a/45986805/15887086

    Parameters
    ----------
    path: matplotlib.path
        polygon path
    Returns
    -------
    isCounterCloc1: int
        1 if the path is counter clockwise, 0 otherwise
    """
    v = path.vertices - path.vertices[0, :]
    a = np.arctan2(v[1:, 1], v[1:, 0])
    isCounterClock = (a[1:] >= a[:-1]).astype(int).mean() >= 0.5
    return isCounterClock

In [ ]:
import geopandas as gpd

# Load the GeoPackage file
file_path = "../data/stubai/avaScen_HochwinterDry5_WorstCase_20260508135932.gpkg"
gdf = gpd.read_file(file_path)

# View the first few rows
gdf.head()

In [ ]:
gdf.describe()

## Test run


In [ ]:
import avalanchers
# Get necessary grid info
def create_file_header(_sim):
    header = {
        "ncols": _sim.dem.shape[1],
        "nrows": _sim.dem.shape[0],
        "xllcenter": _sim.dem_bounds[0],
        "yllcenter": _sim.dem_bounds[2],
        "cellsize": _sim.cell_size,
        "NODATA_value": -9999
    }
    return header

settings = {
    "dem_path": f"../data/stubai/10DTM_pilotStubai.tif"

}
sim = avalanchers.PySimulation.new()
sim.create(settings)

for poly in gdf.geometry[0].geoms:
    exterior_coords = list(poly.exterior.coords)
    # print("Exterior:", exterior_coords)
    x_coords, y_coords = poly.exterior.xy
    line_dict = {
        "x": np.array(x_coords),
        "y": np.array(y_coords)
    }
line_dict

pra = polygon2Raster(create_file_header(sim), line_dict, radius=0.1, th=1.0)

In [ ]:
def release_area_to_png(pra, output_path):
    """Save the release area raster as a PNG image.

    Parameters
    ----------
    pra: 2D numpy array
        The release area raster to be saved.
    output_path: str
        The file path where the PNG image will be saved.
    """
    alpha_channel = (pra * 100).astype(np.uint8)
    height, width = alpha_channel.shape
    rgba_array = np.zeros((height, width, 4), dtype=np.uint8)
    rgba_array[:, :, 0] = 255
    rgba_array[:, :, 3] = alpha_channel
    img = Image.fromarray(rgba_array, mode="RGBA")
    img.save(output_path)

In [ ]:
gdf_single_parts = gdf.geometry.explode(index_parts=False)

for geo in gdf_single_parts.geometry:
    x_coords, y_coords = geo.exterior.xy
    line_dict = {
        "x": np.array(x_coords),
        "y": np.array(y_coords)
    }
    pra += polygon2Raster(create_file_header(sim), line_dict, radius=0.1, th=1.0)
    
# remove overlaps
pra = np.where(pra > 0, 1.0, 0.0)
release_area_to_png(pra, "release_area.png")